In [1]:
#Goal: Build an AI agent that can:
#Take a high-level goal 
#break it into steps 
#call tools 
#generate a structured plan.

In [2]:
#Architecture
#User Input -> Prompt Template (System + User) -> LLM (Planner) -> Tool Calls (APIs / Functions)
# -> Final Plan (Streaming Output)

In [ ]:
#Install dependencies
#pip install openai python-dotenv

In [2]:
#create .env file (Ubuntu)
#--------------------------
#cd multi-step-planning-agent
#touch .env
#nano .env
#Add: OPENAI_API_KEY="your api key"
#CTRL + X
#Y
#Enter

In [10]:
from openai import OpenAI
from prompts import SYSTEM_PROMPT, user_prompt
from tools import search_places, get_weather, estimate_budget
from tools import TOOLS
import config

In [11]:
import json

In [12]:
client = OpenAI()

In [13]:
def handle_tool_call(tool_call):
    name = tool_call.function.name
    args = json.loads(tool_call.function.arguments)
    if name == "search_places":
        return search_places(**args)
    elif name == "get_weather":
        return get_weather(**args)
    elif name == "estimate_budget":
        return estimate_budget(**args)


In [14]:
def run_agent(goal):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt(goal)}
    ]
    loop_count = 0
    while True:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=TOOLS,
            tool_choice="auto"
        )
        msg = response.choices[0].message
        print("msg.tool_calls:", msg.tool_calls)
        #Always append assistant message
        messages.append({
            "role": "assistant",
            "content": msg.content,
            "tool_calls": msg.tool_calls
        })
        #If tool calls exist
        if msg.tool_calls:
            loop_count += 1
            print("Tool call count:", loop_count)
            for tool_call in msg.tool_calls:
                result = handle_tool_call(tool_call)

                messages.append({
                    "role": "tool",
                    "content": result,
                    "tool_call_id": tool_call.id
                })
        else:
            #Final response streaming
            stream = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=messages,
                stream=True
            )
            for chunk in stream:
                if chunk.choices[0].delta.content:
                    print(chunk.choices[0].delta.content, end="")
            break

In [15]:
if __name__ == "__main__":
    goal = "Plan a 3-day trip to Kerala from June 10 to June 12. Provide a rough budget."
    run_agent(goal)

msg.tool_calls: [ChatCompletionMessageFunctionToolCall(id='call_0xe1UhG8YgKG2y3Gg0LJcRYF', function=Function(arguments='{"days":"3"}', name='estimate_budget'), type='function')]
Tool call count: 1
msg.tool_calls: [ChatCompletionMessageFunctionToolCall(id='call_gvddRpK4mxJgFNqFFYnxDQT7', function=Function(arguments='{"location": "Kerala"}', name='search_places'), type='function'), ChatCompletionMessageFunctionToolCall(id='call_xNVW83lZ5AT5V9sYpahoLnID', function=Function(arguments='{"location": "Kerala"}', name='get_weather'), type='function')]
Tool call count: 2
msg.tool_calls: None
Here's a structured plan for your 3-day trip to Kerala from June 10 to June 12:

### Step 1: Estimated Budget
- **Total Estimated Budget**: ₹15,000

### Step 2: Weather Forecast
- **Weather**: 25°C, partly cloudy

### Step 3: Top Places to Visit
1. **Munnar**
   - Known for tea gardens, beautiful landscapes, and pleasant weather.
2. **Alleppey**
   - Famous for its backwaters and houseboat experiences.
3. *